# 06 · E3: cyclist risk per Bike Share trip (RQ3)

**RQ3:** When Bike Share trips are used as a measure of exposure, how does the cyclist collision rate per trip vary with weather conditions?

Negative binomial regression of cyclist collisions per neighbourhood-hour, with log(Bike Share trips in the same neighbourhood and hour) as an offset, so that coefficients describe collisions **per trip**. The model is compared with the same model without exposure. The analysis covers neighbourhoods with Bike Share stations and hours with at least one trip, for 2019–2025 (files with station IDs and consistent timestamp formats).

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.spark_session import load_config, path, get_spark

cfg = load_config()
FIG, TAB = path(cfg, "figures"), path(cfg, "tables")
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
PROC = path(cfg, "processed"); PROC.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "savefig.bbox": "tight", "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
from src import models as M
spark = get_spark(cfg); spark.sparkContext.setLogLevel("ERROR")
cols = ["hood_158", "date", "hour", "trips", "cyclist_collisions", "rain", "snow", "vis_lt1", "vis_1to5",
        "temp_bin", "snow_on_ground", "hours_since_precip", "weather_cat", "dow", "month", "year", "holiday",
        "school_day", "daylight", "covid"]
nh = spark.read.parquet(str(PROC / "neighbourhood_hour")).select(*cols).toPandas()
spark.stop()
nh = nh[nh["temp_bin"] != "nan"]
print(f"{len(nh):,} neighbourhood-hours, {nh['hood_158'].nunique()} neighbourhoods, {nh['trips'].sum():,} trips, {nh['cyclist_collisions'].sum():,} cyclist collisions")

## 1. Rates per 100,000 trips by weather (exact Poisson 95% CI)

In [ ]:
g = nh.groupby("weather_cat").agg(trips=("trips", "sum"), collisions=("cyclist_collisions", "sum"), hours=("trips", "size"))
g["rate"], g["ci_low"], g["ci_high"] = M.poisson_rate_ci(g["collisions"], g["trips"])
g["trips_per_hour"] = g["trips"] / g["hours"]
g.to_csv(TAB / "e3_rates_by_weather.csv")
display(g.round(2))
order = [c for c in ["dry_clear", "rain", "snow", "low_visibility"] if c in g.index]
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].bar(order, g.loc[order, "trips_per_hour"], color="#8c8c8c"); ax[0].set_title("Exposure: trips per neighbourhood-hour")
gg = g.loc[order]
ax[1].errorbar(order, gg["rate"], yerr=[gg["rate"] - gg["ci_low"], gg["ci_high"] - gg["rate"]], fmt="o", capsize=4, color="#C44E52")
ax[1].set_title("Cyclist collisions per 100,000 trips"); ax[1].set_ylim(bottom=0)
fig.savefig(FIG / "fig_e3_rates.png"); plt.show()

## 2. Exposure-adjusted negative binomial vs model without exposure

In [ ]:
terms_w = M.WEATHER_TERMS
terms_c = ["C(hour)", "C(dow)", "C(month)", "C(year)", "holiday", "school_day", "daylight", "covid"]
terms_c = [t for t in terms_c if t.startswith("C(") or nh[t].nunique() > 1]   # drop indicators without variation
f = M.formula("cyclist_collisions", terms_w + terms_c)
with_exp = M.fit_nb(f, nh, np.log(nh["trips"]))
no_exp = M.fit_nb(f, nh)
cmp_ = pd.concat({"per trip (offset log trips)": M.ratio_table(with_exp), "total (no exposure)": M.ratio_table(no_exp)}, axis=1)
cmp_.to_csv(TAB / "e3_irr_with_without_exposure.csv")
cmp_.round(3)

In [ ]:
keys = ["rain", "snow", "vis_lt1", "vis_1to5", "snow_on_ground"]
fig, ax = plt.subplots(figsize=(6, 3.2))
for off, (name, tab), col in zip([-0.12, 0.12], [("per trip", M.ratio_table(with_exp)), ("total", M.ratio_table(no_exp))], ["#C44E52", "#8c8c8c"]):
    t = tab.loc[keys]
    ax.errorbar(t["estimate"], np.arange(len(keys)) + off, xerr=[t["estimate"] - t["ci_low"], t["ci_high"] - t["estimate"]], fmt="o", capsize=3, color=col, label=name)
ax.axvline(1, color="grey", ls="--", lw=0.8); ax.set_yticks(range(len(keys))); ax.set_yticklabels(keys)
ax.set_xlabel("IRR"); ax.set_title("Cyclist collisions: with and without exposure"); ax.legend(frameon=False)
fig.savefig(FIG / "fig_e3_irr.png"); plt.show()

## 3. Hypothesis H3

In [ ]:
tw, tn = M.ratio_table(with_exp), M.ratio_table(no_exp)
h3 = pd.DataFrame([
    {"quantity": "rain: total cyclist collisions (no exposure)", **tn.loc["rain", ["estimate", "ci_low", "ci_high"]].to_dict(), "expected": "less"},
    {"quantity": "rain: cyclist collisions per trip", **tw.loc["rain", ["estimate", "ci_low", "ci_high"]].to_dict(), "expected": "greater"},
]).set_index("quantity")
h3["verdict"] = [M.hypothesis_verdict(r.estimate, r.ci_low, r.ci_high, expected=r.expected) for r in h3.itertuples()]
h3.to_csv(TAB / "h3.csv")
h3.round(3)